In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
df = pd.read_csv("week11_shop.csv", encoding="cp949")
print(df.shape)
df.head()

(30, 8)


,고객ID,나이,연소득,구매금액,구매횟수,성별,등급,지역
0,C001,25.0,3200,150000,3,남,SILVER,서울
1,C002,32.0,4500,280000,7,여,BRONZE,경기
2,C003,NaN,5100,320000,8,남,GOLD,서울
3,C004,45.0,2800,98000,2,여,BRONZE,부산
4,C005,28.0,6200,450000,12,남,GOLD,서울


In [ ]:
df.describe(include="all")

,고객ID,나이,연소득,구매금액,구매횟수,성별,등급,지역
count,30,27.000000,30.000000,30.00000,30.000000,30,30,30
unique,30,NaN,NaN,NaN,NaN,2,4,3
top,C001,NaN,NaN,NaN,NaN,남,SILVER,서울
freq,1,NaN,NaN,NaN,NaN,15,10,15
mean,NaN,72.814815,8196.633333,339200.00000,8.933333,NaN,NaN,NaN
std,NaN,185.355176,17424.918801,197169.52272,5.656448,NaN,NaN,NaN
min,NaN,22.000000,2500.000000,88000.00000,2.000000,NaN,NaN,NaN
25%,NaN,29.000000,3650.000000,163750.00000,4.000000,NaN,NaN,NaN
50%,NaN,36.000000,4850.000000,315000.00000,8.000000,NaN,NaN,NaN
75%,NaN,44.500000,6275.000000,480000.00000,12.750000,NaN,NaN,NaN


In [ ]:
df.describe()

,나이,연소득,구매금액,구매횟수
count,30.000000,30.000000,30.00000,30.000000
mean,37.192308,5031.034483,339200.00000,8.933333
std,9.206517,1731.274884,197169.52272,5.656448
min,22.000000,2500.000000,88000.00000,2.000000
25%,29.500000,3650.000000,163750.00000,4.000000
50%,37.192308,4850.000000,315000.00000,8.000000
75%,41.750000,6125.000000,480000.00000,12.750000
max,61.000000,9200.000000,790000.00000,22.000000


In [ ]:
df.describe(include="object")

,고객ID,성별,등급,지역
count,30,30,30,30
unique,30,2,4,3
top,C001,남,SILVER,서울
freq,1,15,10,15


In [ ]:
df.select_dtypes("number").skew().sort_values(ascending=False)

,0
연소득,5.391163
나이,5.173102
구매횟수,0.657254
구매금액,0.591056


In [ ]:
df.select_dtypes("number").corr()["구매금액"].sort_values(ascending=False)

,구매금액
구매금액,1.000000
구매횟수,0.996237
연소득,0.417728
나이,0.084146


In [ ]:
print(round(df["나이"].corr(df["연소득"]),3))

-0.001


In [ ]:
for col in ["나이","연소득"]:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  upper = Q3 + 1.5 * (Q3 - Q1)
  clean_mean = df.loc[df[col] <= upper, col].mean()
  df[col] = df[col].where(df[col] <= upper).fillna(clean_mean)

df[["나이","연소득"]].agg(["max","mean"]).round(1)
print(round(df["나이"].corr(df["연소득"]),3))

0.706


In [ ]:
corr_mat = df.select_dtypes("number").corr().abs()
corr_mat[(corr_mat > 0.8) & (corr_mat < 1.0)]

,나이,연소득,구매금액,구매횟수
나이,NaN,NaN,0.828449,0.819881
연소득,NaN,NaN,0.934906,0.938549
구매금액,0.828449,0.934906,NaN,0.996237
구매횟수,0.819881,0.938549,0.996237,NaN


In [ ]:
X = df[["나이","연소득","구매금액"]]
vif_df = pd.DataFrame({
    "Feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif_df)

  Feature        VIF
0      나이  19.583416
1     연소득  39.486760
2    구매금액  17.696590


In [ ]:
df["등급별_평균구매"] = df.groupby("등급")["구매금액"].transform("mean")
print(df[df["등급"]=="SILVER"]["등급별_평균구매"].iloc[0])

300000.0


In [ ]:
df.groupby("등급").filter(lambda x: x["구매금액"].mean() >= 300000)

,고객ID,나이,연소득,구매금액,구매횟수,성별,등급,지역,등급별_평균구매
0,C001,25.000000,3200.000000,150000,3,남,SILVER,서울,300000.000000
2,C003,37.192308,5100.000000,320000,8,남,GOLD,서울,465000.000000
4,C005,28.000000,6200.000000,450000,12,남,GOLD,서울,465000.000000
5,C006,61.000000,5031.034483,680000,18,남,VIP,서울,713333.333333
7,C008,35.000000,4200.000000,175000,4,여,SILVER,부산,300000.000000
8,C009,42.000000,7100.000000,520000,15,남,GOLD,서울,465000.000000
10,C011,38.000000,5500.000000,310000,8,남,SILVER,서울,300000.000000
11,C012,55.000000,9200.000000,790000,22,여,VIP,서울,713333.333333
13,C014,37.192308,5800.000000,380000,9,여,SILVER,경기,300000.000000
14,C015,33.000000,4100.000000,240000,6,남,GOLD,서울,465000.000000


In [ ]:
df

,고객ID,나이,연소득,구매금액,구매횟수,성별,등급,지역,등급별_평균구매
0,C001,25.000000,3200.000000,150000,3,남,SILVER,서울,300000.000000
1,C002,32.000000,4500.000000,280000,7,여,BRONZE,경기,146222.222222
2,C003,37.192308,5100.000000,320000,8,남,GOLD,서울,465000.000000
3,C004,45.000000,2800.000000,98000,2,여,BRONZE,부산,146222.222222
4,C005,28.000000,6200.000000,450000,12,남,GOLD,서울,465000.000000
5,C006,61.000000,5031.034483,680000,18,남,VIP,서울,713333.333333
6,C007,37.192308,3800.000000,210000,5,여,BRONZE,경기,146222.222222
7,C008,35.000000,4200.000000,175000,4,여,SILVER,부산,300000.000000
8,C009,42.000000,7100.000000,520000,15,남,GOLD,서울,465000.000000
9,C010,29.000000,3500.000000,130000,3,여,BRONZE,경기,146222.222222


In [ ]:
# 기본 상관계수 행렬 (Pearson)
df.select_dtypes("number").corr()
df.select_dtypes("number").corr(method="pearson")   # 기본값
df.select_dtypes("number").corr(method="spearman")  # 순위 기반 (이상치에 강함)
df.select_dtypes("number").corr(method="kendall")   # 순서 일치 비율

,나이,연소득,구매금액,구매횟수
나이,1.000000,0.682944,0.659988,0.653058
연소득,0.682944,1.000000,0.921839,0.917100
구매금액,0.659988,0.921839,1.000000,0.968570
구매횟수,0.653058,0.917100,0.968570,1.000000


In [ ]:
print(df.select_dtypes("number").corr()["구매금액"].sort_values(ascending=False))

구매금액    1.000000
구매횟수    0.996237
연소득     0.417728
나이      0.084146
Name: 구매금액, dtype: float64


In [ ]:
print(round(df["나이"].corr(df["연소득"]), 3))

-0.001


In [ ]:
for col in ["나이", "연소득"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    upper = Q3 + 1.5 * (Q3 - Q1)
    clean_mean = df.loc[df[col] <= upper, col].mean()
    df[col] = df[col].where(df[col] <= upper).fillna(clean_mean)

df[["나이", "연소득"]].agg(["max", "mean"]).round(1)

,나이,연소득
max,61.0,9200.0
mean,37.2,5031.0


In [ ]:
print(round(df["나이"].corr(df["연소득"]), 3))

0.706


In [ ]:
# 다중공선성 확인 — 상관 0.8 이상 쌍 찾기
corr_mat = df.select_dtypes("number").corr().abs()
corr_mat[(corr_mat > 0.8) & (corr_mat < 1.0)]



,나이,연소득,구매금액,구매횟수
나이,NaN,NaN,0.828449,0.819881
연소득,NaN,NaN,0.934906,0.938549
구매금액,0.828449,0.934906,NaN,0.996237
구매횟수,0.819881,0.938549,0.996237,NaN


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[["나이", "연소득", "구매금액"]]

# VIF 재계산
vif_df = pd.DataFrame({
    "Feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print(vif_df)

  Feature        VIF
0      나이  19.583416
1     연소득  39.486760
2    구매금액  17.696590


In [ ]:
df["등급별_평균구매"] = df.groupby("등급")["구매금액"].transform("mean")
print(df[df["등급"]=="SILVER"]["등급별_평균구매"].iloc[0])

300000.0


In [ ]:
df.groupby("등급").filter(lambda x: x["구매금액"].mean() >= 300000)

,고객ID,나이,연소득,구매금액,구매횟수,성별,등급,지역,등급별_평균구매
0,C001,25.000000,3200.000000,150000,3,남,SILVER,서울,300000.000000
2,C003,37.192308,5100.000000,320000,8,남,GOLD,서울,465000.000000
4,C005,28.000000,6200.000000,450000,12,남,GOLD,서울,465000.000000
5,C006,61.000000,5031.034483,680000,18,남,VIP,서울,713333.333333
7,C008,35.000000,4200.000000,175000,4,여,SILVER,부산,300000.000000
8,C009,42.000000,7100.000000,520000,15,남,GOLD,서울,465000.000000
10,C011,38.000000,5500.000000,310000,8,남,SILVER,서울,300000.000000
11,C012,55.000000,9200.000000,790000,22,여,VIP,서울,713333.333333
13,C014,37.192308,5800.000000,380000,9,여,SILVER,경기,300000.000000
14,C015,33.000000,4100.000000,240000,6,남,GOLD,서울,465000.000000


Slide 15

In [4]:
Q1 = df["연소득"].quantile(0.25)
Q3 = df["연소득"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df["연소득"] < lower) | (df["연소득"] > upper)]
print(outliers)
print(len(outliers))


   고객ID    나이    연소득    구매금액  구매횟수 성별   등급  지역
5  C006  61.0  99999  680000    18  남  VIP  서울
1


In [5]:
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"[{col}] Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}")
    print(f"  하한={lower:.1f}, 상한={upper:.1f}, 이상치={len(outliers)}개")
    return outliers

# 모든 수치형 열에 한번에 적용
for col in df.select_dtypes("number").columns:
    detect_outliers_iqr(df, col)

[나이] Q1=29.0, Q3=44.5, IQR=15.5
  하한=5.8, 상한=67.8, 이상치=1개
[연소득] Q1=3650.0, Q3=6275.0, IQR=2625.0
  하한=-287.5, 상한=10212.5, 이상치=1개
[구매금액] Q1=163750.0, Q3=480000.0, IQR=316250.0
  하한=-310625.0, 상한=954375.0, 이상치=0개
[구매횟수] Q1=4.0, Q3=12.8, IQR=8.8
  하한=-9.1, 상한=25.9, 이상치=0개


In [8]:
# 나이 기준으로 IQR 다시 계산
Q1 = df["나이"].quantile(0.25)
Q3 = df["나이"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df["나이"] = df["나이"].clip(lower=lower, upper=upper)

# 확인
print(df["나이"].max())  # upper 값으로 바뀌었는지 확인
print(df["나이"].min())

67.75
22.0


In [9]:
print(pd.crosstab(df["성별"], df["등급"]))

등급  BRONZE  GOLD  SILVER  VIP
성별                           
남        4     5       5    1
여        5     3       5    2


In [10]:
pd.crosstab(df["성별"], df["등급"], normalize="index")

등급,BRONZE,GOLD,SILVER,VIP
성별,,,,
남,0.266667,0.333333,0.333333,0.066667
여,0.333333,0.200000,0.333333,0.133333


In [11]:
df = pd.read_csv("week11_item.csv", encoding="utf-8-sig")
#1
print(df.shape)

(28, 7)


In [12]:
#2
print(df.isnull().sum())
print(df.isnull().sum().sum())

상품ID    0
카테고리    0
가격      3
평점      0
리뷰수     0
재고      0
할인여부    0
dtype: int64
3


In [13]:
#3
df.describe()

,가격,평점,리뷰수,재고
count,2.500000e+01,28.000000,28.000000,28.000000
mean,4.067300e+06,3.982143,574.357143,95.785714
std,1.998614e+07,0.600033,1859.655747,136.423458
min,5.500000e+03,3.000000,21.000000,1.000000
25%,1.200000e+04,3.475000,64.750000,8.750000
50%,3.800000e+04,4.050000,145.000000,31.500000
75%,1.350000e+05,4.500000,345.250000,127.500000
max,1.000000e+08,4.900000,9999.000000,500.000000


In [14]:
#4
print(df["카테고리"].value_counts())

카테고리
전자기기    10
패션       9
식품       9
Name: count, dtype: int64


In [15]:
#5
print(df.select_dtypes("number").skew().round(3))

가격     5.000
평점    -0.182
리뷰수    5.179
재고     1.812
dtype: float64


In [16]:
#6 detect_outliers_iqr 함수 활용
for col in ["가격", "리뷰수"]:
  detect_outliers_iqr(df, col)

[가격] Q1=12000.0, Q3=135000.0, IQR=123000.0
  하한=-172500.0, 상한=319500.0, 이상치=2개
[리뷰수] Q1=64.8, Q3=345.2, IQR=280.5
  하한=-356.0, 상한=766.0, 이상치=2개


In [ ]:
for col in ["가격", "리뷰수"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    upper = Q3 + 1.5 * (Q3 - Q1)
    print(f"{col}: {(df[col] > upper).sum()}개")

가격: 2개
리뷰수: 2개


In [17]:
#7
for col in ["가격", "리뷰수"]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    upper = Q3 + 1.5 * (Q3 - Q1)
    clean_mean = df.loc[df[col] <= upper, col].mean()
    df[col] = df[col].where(df[col] <= upper).fillna(clean_mean)

print(df[["가격", "리뷰수"]].mean().round(1))

가격     59239.1
리뷰수      202.4
dtype: float64


In [18]:
#8
corr = df.select_dtypes("number").corr()
print(corr["평점"].drop("평점").abs().sort_values(ascending=False).round(3))

리뷰수    0.834
재고     0.826
가격     0.794
Name: 평점, dtype: float64


In [19]:
#9
print(df.groupby("카테고리")["평점"].mean().round(1))

카테고리
식품      3.3
전자기기    4.6
패션      4.1
Name: 평점, dtype: float64


In [ ]:
#10
print(pd.crosstab(df["카테고리"], df["할인여부"], normalize="index").round(3))

할인여부      N      Y
카테고리              
식품    1.000  0.000
전자기기  0.100  0.900
패션    0.222  0.778
